<a href="https://colab.research.google.com/github/yuricod/7688581-Expert-Git-GitHub/blob/main/05_dashboard_plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Visualisation et dashboard

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 2 · 9h45** · Exercice

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Construire des visualisations analytiques et communicantes
- Monter un dashboard interactif Plotly
- Y intégrer la représentativité des segments et des alertes sur les biais

In [1]:
# Configuration du cours conservée ; téléchargement uniquement si nécessaire.
import os, sys, subprocess, importlib.util
for paquet in ["fairlearn", "plotly", "gdown"]:
    if importlib.util.find_spec(paquet) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquet])
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"
if not os.path.exists("config_projet.py"):
    import gdown
    gdown.download(id=CONFIG_ID, output="config_projet.py", quiet=True)
print("✅ Configuration prête")

✅ Configuration prête


In [2]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "commerce"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")
from IPython.display import display


Maison Lumen · 24224 lignes, 16 colonnes


In [3]:
import plotly.express as px
import plotly.graph_objects as go
from config_projet import exporter_html, PALETTE
from fairlearn.metrics import MetricFrame, selection_rate, count
df = nettoyage_minimal(brut, cfg)
for col in cfg["sensibles"]:
    df[col] = df[col].fillna("inconnu")
mensuel = df.set_index(cfg["date"])[cfg["ca"]].resample("MS").sum()

**Application à Maison Lumen.** Je conserve les étapes demandées par le professeur. Les avoirs négatifs sont exclus selon le nettoyage du cours ; les offres et les écarts entre groupes sont étudiés sur les commandes observées.

## 1. Les chiffres clés
Trois indicateurs `go.Indicator` : chiffre d'affaires de 2025 avec l'évolution par rapport à 2024, part de votre segment principal, taux de décision favorable.

In [4]:
ca_annee=mensuel.groupby(mensuel.index.year).sum()
ca24=float(ca_annee.loc[2024]); ca25=float(ca_annee.loc[2025])
par_canal=df.groupby(cfg["segment"])[cfg["ca"]].sum().sort_values(ascending=False)
principal=par_canal.index[0]; part=par_canal.iloc[0]/par_canal.sum()*100
taux_global=df[cfg["decision"]].mean()*100
kpis=go.Figure()
kpis.add_trace(go.Indicator(mode="number+delta",value=ca25,number={"suffix":" €","valueformat":",.0f"},delta={"reference":ca24,"relative":True,"valueformat":".1%"},title={"text":"CA 2025 · évolution / 2024"},domain={"x":[0,.32],"y":[0,1]}))
kpis.add_trace(go.Indicator(mode="number",value=part,number={"suffix":" %","valueformat":".1f"},title={"text":f"Part du CA · {principal}"},domain={"x":[.34,.65],"y":[0,1]}))
kpis.add_trace(go.Indicator(mode="number",value=taux_global,number={"suffix":" %","valueformat":".1f"},title={"text":"Commandes précédées d’une offre"},domain={"x":[.68,1],"y":[0,1]}))
kpis.update_layout(height=240,template="plotly_white")
kpis.show()
print(f"Le CA 2025 atteint {ca25:,.0f} €, soit {(ca25/ca24-1)*100:+.1f} % sur un an. Le canal {principal} représente {part:.1f} % des ventes de toute la période.")

Le CA 2025 atteint 702,326 €, soit +6.7 % sur un an. Le canal magasin représente 49.4 % des ventes de toute la période.


## 2. La tendance, pour les décideurs
Courbe mensuelle, tendance sur 12 mois, et l'événement du brief annoté. Le titre dit la conclusion.

In [5]:
courbe=go.Figure()
courbe.add_trace(go.Scatter(x=mensuel.index,y=mensuel,name="CA mensuel",mode="lines+markers",line={"color":PALETTE[0]}))
courbe.add_trace(go.Scatter(x=mensuel.index,y=mensuel.rolling(12,min_periods=12).mean(),name="Moyenne mobile sur 12 mois",line={"color":PALETTE[1],"width":3}))
evenement=pd.Timestamp(cfg["evenement"][0])
courbe.add_vline(x=evenement.timestamp()*1000,line_dash="dash",line_color=PALETTE[3])
courbe.add_annotation(x=evenement,y=1,yref="paper",text="15 avril 2024 : refonte et lancement de l’app",showarrow=True,ax=45,ay=-35)
courbe.update_layout(title=f"Le CA augmente de {(ca25/ca24-1)*100:.1f} % en 2025, après une forte progression en 2024",xaxis_title="Mois",yaxis_title="CA TTC des commandes retenues (€)",template="plotly_white",margin={"t":100})
courbe.show()
print("Les pics de fin d’année reviennent régulièrement. La moyenne sur 12 mois aide à lire la tendance ; elle réagit avec retard. La date annotée ne démontre pas que la refonte explique la croissance.")

Les pics de fin d’année reviennent régulièrement. La moyenne sur 12 mois aide à lire la tendance ; elle réagit avec retard. La date annotée ne démontre pas que la refonte explique la croissance.


## 3. L'analyse, pour l'équipe data
Un graphique plus riche : chiffre d'affaires mensuel par segment, avec un sélecteur interactif.

In [6]:
segments=df.groupby([pd.Grouper(key=cfg["date"],freq="MS"),cfg["segment"]])[cfg["ca"]].sum().unstack(fill_value=0)
detail=go.Figure()
for i,col in enumerate(segments.columns):
    detail.add_trace(go.Scatter(x=segments.index,y=segments[col],name=col,mode="lines+markers",line={"color":PALETTE[i]}))
boutons=[{"label":"Tous les canaux","method":"update","args":[{"visible":[True]*len(segments.columns)}]}]
for i,col in enumerate(segments.columns):
    boutons.append({"label":col,"method":"update","args":[{"visible":[j==i for j in range(len(segments.columns))]}]})
detail.update_layout(title=f"{principal.capitalize()} reste le premier canal sur l’ensemble de la période",xaxis_title="Mois",yaxis_title="CA (€)",template="plotly_white",updatemenus=[{"buttons":boutons,"x":1,"y":1.2}])
detail.show()
print("Le sélecteur permet de suivre chaque canal sans confondre une hausse en euros et une hausse de part de marché. L’app est déjà présente avant son lancement annoncé : ce point du jeu pédagogique reste à clarifier.")

Le sélecteur permet de suivre chaque canal sans confondre une hausse en euros et une hausse de part de marché. L’app est déjà présente avant son lancement annoncé : ce point du jeu pédagogique reste à clarifier.


## 4. L'indicateur de représentativité
Pour chaque variable sensible, le ratio part observée ÷ part de référence, coloré selon le statut, avec les seuils 0,8 et 1,25.

In [7]:
clients=df.sort_values(cfg["date"]).drop_duplicates("client_id")
ref=reference()
lignes=[]
for col in cfg["sensibles"]:
    obs=clients[col].value_counts(normalize=True).rename("part_observee")
    r=ref.loc[ref.variable.eq(col)].set_index("modalite").join(obs).reset_index()
    r["ratio"]=r.part_observee/r.part_reference
    r["statut"]=np.select([r.ratio.lt(.8),r.ratio.gt(1.25)],["Sous-représenté","Surreprésenté"],default="Dans les seuils")
    lignes.append(r)
representativite=pd.concat(lignes,ignore_index=True)
rep=px.bar(representativite,x="modalite",y="ratio",color="statut",facet_row="variable",category_orders={"variable":cfg["sensibles"]},color_discrete_map={"Sous-représenté":PALETTE[5],"Surreprésenté":PALETTE[0],"Dans les seuils":PALETTE[2]},hover_data=["part_observee","part_reference"],height=900,title="Les jeunes clients sont surreprésentés ; les plus âgés le sont moins")
rep.update_xaxes(matches=None,title_text="Groupe",showticklabels=True)
rep.update_yaxes(title_text="Part observée / référence")
rep.add_hline(y=.8,line_dash="dash",line_color="black")
rep.add_hline(y=1.25,line_dash="dot",line_color="black")
rep.update_layout(template="plotly_white")
display(representativite.round(3));rep.show()
print("Chaque client compte une fois. La référence est une approximation pédagogique de la population adulte, pas la clientèle cible exacte de l’enseigne. Les régions inconnues restent dans le dénominateur, sans ratio de référence inventé.")

,modalite,variable,part_reference,part_observee,ratio,statut
0,18-24,tranche_age,0.10,0.139,1.389,Surreprésenté
1,25-34,tranche_age,0.15,0.295,1.967,Surreprésenté
2,35-44,tranche_age,0.16,0.253,1.580,Surreprésenté
3,45-54,tranche_age,0.17,0.164,0.965,Dans les seuils
4,55-64,tranche_age,0.16,0.087,0.545,Sous-représenté
5,65+,tranche_age,0.26,0.062,0.239,Sous-représenté
6,F,genre,0.52,0.624,1.199,Dans les seuils
7,H,genre,0.48,0.376,0.784,Sous-représenté
8,Île-de-France,region,0.19,0.310,1.629,Surreprésenté
9,Nord-Est,region,0.24,0.168,0.701,Sous-représenté


Chaque client compte une fois. La référence est une approximation pédagogique de la population adulte, pas la clientèle cible exacte de l’enseigne. Les régions inconnues restent dans le dénominateur, sans ratio de référence inventé.


## 5. La parité de la décision
Taux de décision favorable par groupe, avec la ligne des quatre cinquièmes du meilleur taux.

In [8]:
groupes=[]
for col in cfg["sensibles"]:
    mf=MetricFrame(metrics={"taux":selection_rate,"n":count},y_true=df[cfg["resultat"]],y_pred=df[cfg["decision"]],sensitive_features=df[col])
    t=mf.by_group.reset_index().rename(columns={col:"modalite"})
    t["variable"]=col;t["seuil"]=.8*t.taux.max()
    t["alerte"]=t.taux.lt(t.seuil)
    groupes.append(t)
parite_table=pd.concat(groupes,ignore_index=True)
parite_table["taux_pct"]=parite_table.taux*100
parite=px.bar(parite_table,x="modalite",y="taux_pct",facet_row="variable",category_orders={"variable":cfg["sensibles"]},color="alerte",color_discrete_map={True:PALETTE[5],False:PALETTE[0]},hover_data=["n"],height=900,title="Les clients âgés reçoivent moins souvent une offre avant leur commande")
parite.update_xaxes(matches=None,showticklabels=True,title_text="Groupe")
parite.update_yaxes(title_text="Commandes précédées d’une offre (%)")
for i,col in enumerate(cfg["sensibles"]):
    seuil=parite_table.loc[parite_table.variable.eq(col),"seuil"].iloc[0]*100
    parite.add_hline(y=seuil,row=len(cfg["sensibles"])-i,col=1,line_dash="dash",line_color="black")
parite.update_layout(template="plotly_white",legend_title_text="Sous le seuil des 4/5")
display(parite_table);parite.show()
print("Le seuil est calculé séparément pour chaque variable : 80 % du meilleur taux. Il sert à repérer les écarts, sans constituer une preuve de discrimination. Les taux portent sur les commandes, les clients fréquents pèsent donc davantage.")

,modalite,taux,n,variable,seuil,alerte,taux_pct
0,18-24,0.477915,3396.0,tranche_age,0.387028,False,47.791519
1,25-34,0.483785,7123.0,tranche_age,0.387028,False,48.378492
2,35-44,0.384951,5954.0,tranche_age,0.387028,True,38.495129
3,45-54,0.401703,4110.0,tranche_age,0.387028,False,40.170316
4,55-64,0.247549,2040.0,tranche_age,0.387028,True,24.754902
5,65+,0.173943,1466.0,tranche_age,0.387028,True,17.394270
6,F,0.420589,15105.0,genre,0.336471,False,42.058921
7,H,0.380565,8984.0,genre,0.336471,False,38.056545
8,Nord-Est,0.403491,4010.0,region,0.331370,False,40.349127
9,Ouest,0.405405,4292.0,region,0.331370,False,40.540541


Le seuil est calculé séparément pour chaque variable : 80 % du meilleur taux. Il sert à repérer les écarts, sans constituer une preuve de discrimination. Les taux portent sur les commandes, les clients fréquents pèsent donc davantage.


## 6. Les alertes
Un bloc HTML qui liste en clair les groupes sous-représentés et les groupes sous la ligne des 4/5. Ne comptez pas sur la couleur seule : écrivez-le.

In [9]:
from html import escape
sous=representativite.loc[representativite.ratio.lt(.8)]
faibles=parite_table.loc[parite_table.alerte]
a="".join(f"<li>{escape(r.variable)} — {escape(str(r.modalite))} : ratio {r.ratio:.2f}, sous le seuil de 0,80.</li>" for r in sous.itertuples())
b="".join(f"<li>{escape(r.variable)} — {escape(str(r.modalite))} : offres {r.taux:.1%}, seuil {r.seuil:.1%}, {int(r.n)} commandes.</li>" for r in faibles.itertuples())
alertes=f"<section class='alerte'><h2>Points à vérifier</h2><h3>Clients sous-représentés</h3><ul>{a or '<li>Aucun groupe sous le seuil.</li>'}</ul><h3>Accès aux offres sous la ligne des quatre cinquièmes</h3><ul>{b or '<li>Aucun groupe sous le seuil.</li>'}</ul><p>Ces signaux demandent une vérification des règles de ciblage et du profil des clients. Ils ne démontrent pas une discrimination.</p></section>"
display(__import__("IPython.display",fromlist=["HTML"]).HTML(alertes))

## 7. L'export HTML autonome
Assemblez le tout avec `exporter_html` (fourni) : une page pour les décideurs, une annexe pour l'équipe data. Chaque figure reçoit une description courte, utile aux lecteurs d'écran.

In [10]:
contexte="""<section><h2>Périmètre et lecture</h2><p>Données fictives : commandes de janvier 2023 à décembre 2025. Doublons exacts et avoirs négatifs exclus conformément au cours. Le total est un CA TTC des ventes retenues, pas un CA comptable net des avoirs. Les remises sont déjà déduites des montants.</p><p>Une offre est un envoi dans le mois précédent la commande. Le fichier n’observe pas tous les clients sans achat. La représentativité compte les clients uniques ; la parité compte les commandes.</p></section>"""
recommandations="""<section><h2>Ce que je proposerais à la direction</h2><p>Suivre le CA et le panier moyen par canal, en comparant les mêmes mois d’une année à l’autre. Vérifier les règles de ciblage des offres pour les clients âgés, puis comparer des profils proches à canal égal. Compléter ces tableaux par le taux de retour, la satisfaction et les écarts d’accès aux offres.</p><p>La refonte est un repère temporel, pas une preuve de causalité. Le fichier ne contient pas d’émissions ou de données de livraison permettant de calculer un indicateur environnemental fiable.</p></section>"""
blocs=[contexte,("Chiffres clés",kpis,"CA 2025 et évolution sur un an, part du principal canal, taux global d’offre."),("Tendance",courbe,"CA mensuel et moyenne mobile sur douze mois ; refonte du 15 avril 2024 indiquée."),("Représentativité",rep,"Ratios des clients uniques à la référence pédagogique par âge, genre et région ; seuils 0,8 et 1,25."),("Accès aux offres",parite,"Taux par groupe et seuil de quatre cinquièmes du meilleur taux, pour chaque variable."),alertes,recommandations]
taille=exporter_html(f"{cfg['nom']} · tableau de bord",blocs,f"dashboard_{SECTEUR}.html")
annexe=[contexte,("CA par canal",detail,"Sélection interactive de tous les canaux ou d’un seul ; CA mensuel en euros."),"<section><h2>Table des ratios</h2>"+representativite.to_html(index=False,float_format=lambda x:f"{x:.3f}")+"</section>","<section><h2>Table des offres et effectifs</h2>"+parite_table.to_html(index=False,float_format=lambda x:f"{x:.3f}")+"</section>",alertes]
taille_annexe=exporter_html(f"{cfg['nom']} · annexe data",annexe,f"annexe_data_{SECTEUR}.html")
print(f"dashboard_{SECTEUR}.html : {taille:.1f} Mo")
print(f"annexe_data_{SECTEUR}.html : {taille_annexe:.1f} Mo")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(f"dashboard_{SECTEUR}.html")
    files.download(f"annexe_data_{SECTEUR}.html")

dashboard_commerce.html : 4.6 Mo
annexe_data_commerce.html : 4.6 Mo


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Check-list du livrable
- [ ] Le fichier s'ouvre seul dans un navigateur, sans connexion.
- [ ] Chaque graphique a un titre qui conclut et une description.
- [ ] Les alertes sont écrites, pas seulement colorées.
- [ ] Les couleurs restent lisibles pour un daltonien (palette Okabe-Ito).

💾 Enregistrez le notebook dans GitHub (« Dashboard Plotly exporté ») et envoyez le fichier HTML téléchargé au formateur.